### **Welcome Reef Rangers!! This is the code scripts used for ingesting all the different global open sourced data into your Lakehouse.**

In [ ]:
# Step 1: Import libraries
import requests
# Step 1a: If xarray not present, create a new environment,
  # Browse to libraries, add external Libraries and add xarray and Publish environment
   # Run this notebook in new environment with xarray library installed
import xarray as xr 
import pandas as pd
import os as os
from pyspark.sql import SparkSession
%pip install netCDF4 h5netcdf



StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:
from datetime import datetime, timedelta
import requests
import os
import xarray as xr

base_url = "https://www.ncei.noaa.gov/data/sea-surface-temperature-optimum-interpolation/v2.1/access/avhrr"
days_back = 365
downloaded_files = []

for i in range(days_back):
    target_date = datetime.utcnow() - timedelta(days=i)
    year_month = target_date.strftime("%Y%m")
    date_str   = target_date.strftime("%Y%m%d")
    file_name  = f"oisst-avhrr-v02r01.{date_str}.nc"
    url        = f"{base_url}/{year_month}/{file_name}"
    local_file = f"/lakehouse/default/Files/ReefRangersData/{file_name}"

    # ✅ Skip if file already exists
    if os.path.exists(local_file):
        downloaded_files.append(local_file)
        continue

    # Try download
    response = requests.get(url)
    if response.status_code == 200:
        with open(local_file, "wb") as f:
            f.write(response.content)
        print("✅ Downloaded:", file_name)
        downloaded_files.append(local_file)
    else:
        print("⚠️ Not found:", url)

print("Total files ready:", len(downloaded_files))


StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
import xarray as xr
import pandas as pd
import os

datasets = []

for local_file in downloaded_files:   # from Cell 1
    try:
        ds = xr.open_dataset(local_file, engine="netcdf4")
        datasets.append(ds)
    except Exception as e:
        print("❌ Error opening:", local_file, e)

if datasets:
    # Merge along time dimension
    combined = xr.concat(datasets, dim="time")
    df = combined.to_dataframe().reset_index()

    # Convert to Spark DataFrame
    spark_df = spark.createDataFrame(df)

    # ✅ Deduplicate BEFORE writing
    spark_df = spark_df.dropDuplicates(["time", "lat", "lon"])

    # Append new rows into Delta table
    spark_df.write.format("delta").mode("append").saveAsTable("NOAA_SST")

    print("✅ Appended", len(datasets), "files into Delta table NOAA_SST")
else:
    print("No datasets to merge.")


StatementMeta(, , -1, Cancelled, , Cancelled, True)